## 1. Install matching dependencies

In [ ]:
%pip -q install "scikit-learn==1.6.1" "imbalanced-learn>=0.13,<0.15" openpyxl

## 2. Imports and reproducibility

In [ ]:
import os
import json
import time
import zipfile
import warnings
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib

from scipy.stats import binomtest
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, classification_report
)
from sklearn.preprocessing import label_binarize
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
if sklearn.__version__ != "1.6.1":
    raise RuntimeError(
        "Expected scikit-learn 1.6.1 to match the previous experiment. "
        f"Current version is {sklearn.__version__}."
    )

scikit-learn: 1.6.1
NumPy: 2.4.6
Pandas: 2.2.3


## 3. Upload the dataset and both previous result ZIP files

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

excel_files = [n for n in uploaded_names if n.lower().endswith((".xlsx", ".xls"))]
zip_files = [n for n in uploaded_names if n.lower().endswith(".zip")]

if not excel_files:
    raise FileNotFoundError("Upload Dry_Bean_Dataset.xlsx.")
if len(zip_files) < 2:
    raise FileNotFoundError(
        "Upload both dry_bean_results.zip and mlp_convergence_results.zip."
    )

DATA_PATH = excel_files[0]

original_zip = next(
    (n for n in zip_files if "mlp_convergence" not in n.lower()),
    zip_files[0]
)
mlp_zip = next(
    (n for n in zip_files if "mlp_convergence" in n.lower()),
    zip_files[1]
)

print("Dataset:", DATA_PATH)
print("Original experiment ZIP:", original_zip)
print("MLP convergence ZIP:", mlp_zip)

Saving Dry_Bean_Dataset.xlsx to Dry_Bean_Dataset.xlsx
Saving dry_bean_results.zip to dry_bean_results.zip
Saving mlp_convergence_results.zip to mlp_convergence_results.zip
Dataset: Dry_Bean_Dataset.xlsx
Original experiment ZIP: dry_bean_results.zip
MLP convergence ZIP: mlp_convergence_results.zip


## 4. Extract previous results and load selected hyperparameters

In [ ]:
BASE_DIR = Path("final_verification_inputs")
BASE_DIR.mkdir(exist_ok=True)

ORIGINAL_DIR = BASE_DIR / "original"
MLP_DIR = BASE_DIR / "mlp_convergence"
ORIGINAL_DIR.mkdir(exist_ok=True)
MLP_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(original_zip, "r") as zf:
    zf.extractall(ORIGINAL_DIR)
with zipfile.ZipFile(mlp_zip, "r") as zf:
    zf.extractall(MLP_DIR)

original_root = ORIGINAL_DIR / "dry_bean_results"
mlp_root = MLP_DIR / "mlp_convergence_check"

with open(original_root / "experiment_metadata.json", "r") as f:
    original_metadata = json.load(f)
with open(original_root / "tables" / "best_hyperparameters.json", "r") as f:
    best_params = json.load(f)

original_results = pd.read_csv(
    original_root / "tables" / "ten_model_results.csv"
)
mlp_metrics_previous = pd.read_csv(
    mlp_root / "mlp_convergence_test_metrics.csv"
).iloc[0]

mlp_params = best_params["Multi-Layer Perceptron"]
svm_params = best_params["SVM (RBF)"]

mlp_model_path = mlp_root / "mlp_convergence_checked_model.joblib"
if not mlp_model_path.exists():
    raise FileNotFoundError("Saved convergence-checked MLP model is missing.")

mlp_model = joblib.load(mlp_model_path)

print("MLP selected parameters:", mlp_params)
print("SVM selected parameters:", svm_params)
print("MLP model iterations:", mlp_model.named_steps["model"].n_iter_)

MLP selected parameters: {'select__k': 16, 'model__learning_rate_init': 0.001, 'model__hidden_layer_sizes': [100], 'model__alpha': 0.001}
SVM selected parameters: {'select__k': 16, 'model__gamma': 'scale', 'model__C': 1.0}
MLP model iterations: 407


## 5. Recreate the exact cleaned dataset and stratified split

In [ ]:
excel = pd.ExcelFile(DATA_PATH)
df = None

for sheet_name in excel.sheet_names:
    candidate = pd.read_excel(DATA_PATH, sheet_name=sheet_name)
    if "Class" in candidate.columns:
        df = candidate
        print("Loaded worksheet:", sheet_name)
        break

if df is None:
    raise ValueError("No worksheet containing target column 'Class' was found.")

original_rows = len(df)
duplicate_count = int(df.duplicated().sum())
df_clean = df.drop_duplicates().reset_index(drop=True).copy()

feature_cols = (
    df_clean.drop(columns=["Class"])
    .select_dtypes(include=np.number)
    .columns.tolist()
)

X = df_clean[feature_cols].copy()
y = df_clean["Class"].astype(str).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

assert len(df_clean) == int(original_metadata["rows_after_duplicate_removal"])
assert len(X_train) == int(original_metadata["train_rows"])
assert len(X_test) == int(original_metadata["test_rows"])
assert len(feature_cols) == int(original_metadata["feature_count"])

print("Original rows:", original_rows)
print("Duplicates removed:", duplicate_count)
print("Cleaned rows:", len(df_clean))
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Dataset and split match the original experiment.")

Loaded worksheet: Dry_Beans_Dataset
Original rows: 13611
Duplicates removed: 68
Cleaned rows: 13543
Train shape: (10834, 16)
Test shape: (2709, 16)
Dataset and split match the original experiment.


## 6. Re-run only the MLP five-fold cross-validation at 600 iterations


In [ ]:
MI_SCORE_FUNC = partial(
    mutual_info_classif,
    random_state=RANDOM_STATE,
    discrete_features=False
)

def build_mlp_pipeline(max_iter=600):
    model = MLPClassifier(
        max_iter=max_iter,
        early_stopping=False,
        random_state=RANDOM_STATE
    )
    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=MI_SCORE_FUNC, k=12)),
        ("sampler", SMOTE(random_state=RANDOM_STATE)),
        ("model", model)
    ])
    pipe.set_params(**mlp_params)
    pipe.set_params(
        model__max_iter=max_iter,
        model__early_stopping=False,
        model__random_state=RANDOM_STATE
    )
    return pipe

MAX_ITER = 600
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

mlp_cv_pipeline = build_mlp_pipeline(MAX_ITER)

cv_warning_messages = []
start = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mlp_cv = cross_validate(
        mlp_cv_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        return_train_score=False,
        return_estimator=True,
        n_jobs=1,
        error_score="raise"
    )
    cv_warning_messages = [
        str(w.message) for w in caught
        if issubclass(w.category, ConvergenceWarning)
    ]

cv_elapsed = time.perf_counter() - start

fold_rows = []
for fold_index, estimator in enumerate(mlp_cv["estimator"], start=1):
    fitted_model = estimator.named_steps["model"]
    fold_rows.append({
        "fold": fold_index,
        "macro_f1": float(mlp_cv["test_score"][fold_index - 1]),
        "actual_iterations": int(fitted_model.n_iter_),
        "final_training_loss": float(fitted_model.loss_),
        "reached_iteration_limit": bool(fitted_model.n_iter_ >= MAX_ITER)
    })

fold_results = pd.DataFrame(fold_rows)
cv_mean = float(np.mean(mlp_cv["test_score"]))
cv_std = float(np.std(mlp_cv["test_score"], ddof=1))

print(f"Five-fold CV macro-F1: {cv_mean:.6f} ± {cv_std:.6f}")
print("Total CV time (seconds):", round(cv_elapsed, 2))
print("Convergence warnings captured:", len(cv_warning_messages))
display(fold_results)

Five-fold CV macro-F1: 0.940268 ± 0.005489
Total CV time (seconds): 133.35
Convergence warnings captured: 0


,fold,macro_f1,actual_iterations,final_training_loss,reached_iteration_limit
0,1,0.941187,484,0.093005,False
1,2,0.937481,306,0.093170,False
2,3,0.935365,371,0.097021,False
3,4,0.937955,331,0.094338,False
4,5,0.949352,264,0.104538,False


## 7. Verify the saved convergence-checked MLP on the untouched test set

The saved MLP was trained on the full training partition with the 600-iteration limit and stopped after 407 iterations. This cell reuses that model rather than fitting it again.

In [ ]:
mlp_pred = mlp_model.predict(X_test)
mlp_prob = mlp_model.predict_proba(X_test)
classes = np.array(sorted(y_train.unique()))
y_test_bin = label_binarize(y_test, classes=classes)

def calculate_metrics(y_true, y_pred, y_prob, labels):
    y_true_bin = label_binarize(y_true, classes=labels)
    return {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "test_precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "test_recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "test_recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "test_f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "test_f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "test_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "test_mcc": matthews_corrcoef(y_true, y_pred),
        "test_roc_auc_ovr_macro": roc_auc_score(
            y_true, y_prob, labels=labels, multi_class="ovr", average="macro"
        ),
        "test_pr_auc_macro": average_precision_score(
            y_true_bin, y_prob, average="macro"
        )
    }

mlp_test_metrics = calculate_metrics(y_test, mlp_pred, mlp_prob, classes)
print("Updated MLP test metrics:")
display(pd.Series(mlp_test_metrics).to_frame("value"))

print("MLP saved model iterations:", mlp_model.named_steps["model"].n_iter_)
print("MLP saved model final loss:", mlp_model.named_steps["model"].loss_)

Updated MLP test metrics:


,value
test_accuracy,0.919528
test_precision_macro,0.930356
test_precision_weighted,0.919402
test_recall_macro,0.932324
test_recall_weighted,0.919528
test_f1_macro,0.931182
test_f1_weighted,0.919292
test_balanced_accuracy,0.932324
test_mcc,0.902805
test_roc_auc_ovr_macro,0.994408


MLP saved model iterations: 407
MLP saved model final loss: 0.09960437992398326


## 8. Fit the selected SVM once using its already-selected hyperparameters



In [ ]:
svm_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(score_func=MI_SCORE_FUNC, k=12)),
    ("sampler", "passthrough"),
    ("model", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])
svm_pipeline.set_params(**svm_params)

svm_start = time.perf_counter()
svm_pipeline.fit(X_train, y_train)
svm_fit_seconds = time.perf_counter() - svm_start

svm_pred = svm_pipeline.predict(X_test)
svm_prob_raw = svm_pipeline.predict_proba(X_test)

# Align probability columns to the same sorted class order.
svm_prob = np.zeros((len(X_test), len(classes)))
for j, label in enumerate(svm_pipeline.classes_):
    svm_prob[:, np.where(classes == label)[0][0]] = svm_prob_raw[:, j]

svm_test_metrics = calculate_metrics(y_test, svm_pred, svm_prob, classes)

print("SVM fit time (seconds):", round(svm_fit_seconds, 2))
print("SVM test metrics:")
display(pd.Series(svm_test_metrics).to_frame("value"))

SVM fit time (seconds): 5.65
SVM test metrics:


,value
test_accuracy,0.925803
test_precision_macro,0.935799
test_precision_weighted,0.926865
test_recall_macro,0.937669
test_recall_weighted,0.925803
test_f1_macro,0.936500
test_f1_weighted,0.925987
test_balanced_accuracy,0.937669
test_mcc,0.910555
test_roc_auc_ovr_macro,0.994483


## 9. Recalculate McNemar's exact test for updated MLP versus SVM



In [ ]:
mlp_correct = (mlp_pred == y_test.to_numpy())
svm_correct = (svm_pred == y_test.to_numpy())

mlp_correct_svm_wrong = int(np.sum(mlp_correct & ~svm_correct))
mlp_wrong_svm_correct = int(np.sum(~mlp_correct & svm_correct))
discordant_pairs = mlp_correct_svm_wrong + mlp_wrong_svm_correct

if discordant_pairs == 0:
    mcnemar_p_value = 1.0
else:
    mcnemar_p_value = binomtest(
        min(mlp_correct_svm_wrong, mlp_wrong_svm_correct),
        n=discordant_pairs,
        p=0.5,
        alternative="two-sided"
    ).pvalue

alpha = 0.05
mcnemar_result = {
    "model_A": "Convergence-checked MLP (600 max_iter)",
    "model_B": "SVM (RBF)",
    "A_correct_B_wrong": mlp_correct_svm_wrong,
    "A_wrong_B_correct": mlp_wrong_svm_correct,
    "discordant_pairs": discordant_pairs,
    "exact_two_sided_p_value": float(mcnemar_p_value),
    "alpha": alpha,
    "significant_at_0.05": bool(mcnemar_p_value < alpha)
}

display(pd.Series(mcnemar_result).to_frame("value"))

if mcnemar_p_value < alpha:
    print("Conclusion: the models have significantly different error rates at alpha = 0.05.")
else:
    print("Conclusion: no statistically significant difference in error rates at alpha = 0.05.")

,value
model_A,Convergence-checked MLP (600 max_iter)
model_B,SVM (RBF)
A_correct_B_wrong,35
A_wrong_B_correct,52
discordant_pairs,87
exact_two_sided_p_value,0.085689
alpha,0.05
significant_at_0.05,False


Conclusion: no statistically significant difference in error rates at alpha = 0.05.


## 10. Final comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "MLP (600 max_iter)",
        "CV_macro_F1_mean": cv_mean,
        "CV_macro_F1_std": cv_std,
        **mlp_test_metrics
    },
    {
        "model": "SVM (RBF)",
        "CV_macro_F1_mean": np.nan,
        "CV_macro_F1_std": np.nan,
        **svm_test_metrics
    }
])

display(comparison.round(4))
print("Note: SVM CV values are omitted here because this verification reruns CV only for MLP.")

,model,CV_macro_F1_mean,CV_macro_F1_std,test_accuracy,test_precision_macro,test_precision_weighted,test_recall_macro,test_recall_weighted,test_f1_macro,test_f1_weighted,test_balanced_accuracy,test_mcc,test_roc_auc_ovr_macro,test_pr_auc_macro
0,MLP (600 max_iter),0.9403,0.0055,0.9195,0.9304,0.9194,0.9323,0.9195,0.9312,0.9193,0.9323,0.9028,0.9944,0.9777
1,SVM (RBF),NaN,NaN,0.9258,0.9358,0.9269,0.9377,0.9258,0.9365,0.9260,0.9377,0.9106,0.9945,0.9785


Note: SVM CV values are omitted here because this verification reruns CV only for MLP.


## 11. Save the final verification package

In [ ]:
OUTPUT_DIR = Path("dry_bean_final_verification")
OUTPUT_DIR.mkdir(exist_ok=True)

fold_results.to_csv(OUTPUT_DIR / "mlp_600_iteration_cv_fold_results.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "updated_mlp_vs_svm_metrics.csv", index=False)

with open(OUTPUT_DIR / "mlp_cv_summary.json", "w") as f:
    json.dump({
        "cv_folds": 5,
        "cv_mean_macro_f1": cv_mean,
        "cv_std_macro_f1": cv_std,
        "cv_total_time_seconds": cv_elapsed,
        "max_iter": MAX_ITER,
        "fold_results": fold_rows,
        "convergence_warning_messages": cv_warning_messages
    }, f, indent=2)

with open(OUTPUT_DIR / "updated_mcnemar_test.json", "w") as f:
    json.dump(mcnemar_result, f, indent=2)

with open(OUTPUT_DIR / "final_verification_metadata.json", "w") as f:
    json.dump({
        "rows_after_duplicate_removal": int(len(df_clean)),
        "duplicates_removed": int(duplicate_count),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "random_state": RANDOM_STATE,
        "sklearn_version": sklearn.__version__,
        "mlp_hyperparameters": mlp_params,
        "svm_hyperparameters": svm_params,
        "mlp_test_metrics": mlp_test_metrics,
        "svm_test_metrics": svm_test_metrics,
        "mcnemar": mcnemar_result
    }, f, indent=2, default=str)

# Save the final SVM model for reproducibility; the MLP model is retained from its prior ZIP.
joblib.dump(svm_pipeline, OUTPUT_DIR / "svm_rbf_final_refit.joblib")

zip_path = Path("dry_bean_final_verification.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR.parent))

print("Created:", zip_path.resolve())
files.download(str(zip_path))

Created: /content/dry_bean_final_verification.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>